# Регистрация фаз 4D-КТ и контроль поля деформации

## Исследовательский вопрос

Проверяется, может ли алгоритм `RegisterImagesGreedy` из MONAI Physio перенести форму сердца из фазы 00 в остальные фазы 4D-КТ при сохранении допустимой геометрии. Регистрация строится по интенсивностям КТ. Отдельная автоматическая маска TotalSegmentator v2 (TSv2) целевой фазы используется **после расчёта поля** как контроль совпадения, а не как цель подгонки. Такой контроль независим от поля регистрации, но не является ручной проверкой истинной анатомии.

Основной индивидуальный опыт выполнен на **Нике**: сначала две пары на исходной сетке, затем все **11 переходов** из фазы 00. Ранний опыт на **Георге** (две пары на уменьшенной и исходной сетках) сохранён ниже как исторический методический пилот. Результаты разных людей, сеток и настроек сглаживания не объединяются в одну оценку точности.

Dice — перекрытие бинарных масок, ASSD — среднее симметрическое расстояние их поверхностей, HD95 — 95-й процентиль симметрических расстояний. Определитель якобиана `det J` описывает локальное изменение объёма отображения; неположительное значение на проверенном узле указывает на складку или потерю ориентации. Положительность на дискретных узлах не гарантирует её между ними.

[Код парной регистрации](mp03_04_monai_pair.py), [пофазный отчёт Ника](mp03_nix_full_cycle_report.md), [отчёт по Георгу](mp03_04_report.md) и [следующий этап переноса поверхности](21.16_MP04_Согласованная_4D_геометрия.ipynb) фиксируют происхождение результатов.

## Исходное разрешение: пары Ника 00→01 и 00→07

КТ и автоматические маски `whole_heart` Ника имеют общую сетку `512×512×256` с шагом `0,765625×0,765625×0,625 мм`. Для регистрации использована исходная плотность вокселей. Область расчёта `216×245×250` выбрана **только по маске 00** с запасом 25 мм; целевая маска не определяла границы этой области (crop) и не входила в функцию регистрации. Миллиметры и NIfTI RAS проверены по манифесту конверсии при `unknown` в заголовке единиц; поля ITK заданы в LPS, мм. Фазам не назначены принятые фазы конца диастолы и систолы (ED/ES); дыхательное состояние не подтверждено.

| Поле и вариант | Dice к отдельной маске: неподвижно → поле | ASSD, мм: неподвижно → поле | Минимум `det J` | Решение |
|---|---:|---:|---:|---|
| 00→01, исходное сглаживание | 0,93002 → 0,95304 | 1,395 → 0,888 | 0,179 | технический кандидат |
| 00→07, исходное сглаживание | 0,92654 → 0,96280 | 1,538 → 0,745 | −0,082 | отклонён: 115 неположительных узлов |
| 00→07, усиленное сглаживание | 0,92654 → 0,95905 | 1,538 → 0,826 | 0,547 | технический кандидат |

Усиленное сглаживание устранило найденные складки на узлах проверенного crop ценой небольшого ухудшения Dice относительно отвергнутого варианта. Повтор 00→01 с теми же настройками дал близкий, но не побитово одинаковый результат. Таблица описывает **внутреннее согласие с автоматической целевой маской**, а не точность движения сердца.

Маска `whole_heart` касается верхнего края КТ, потому что анатомический охват 4D-серии меньше статической. Отдельные `blood_pool` и `myocardium` верхней грани не достигают. Это касание само по себе не считается ошибкой или запретом расчёта в наблюдаемой области. [Полный отчёт двух пар](mp03_nix_fullres_report.md), [проверка охвата](mp03_nix_fov_contact_report.md) и [инвентаризация референса](mp03_nix_reference_inventory.md) содержат QC и границы интерпретации.

## Все целевые фазы Ника

На исходной сетке выполнены **11 пар `phase_00→phase_01…11`** с одинаковым сглаживанием `3.0vox 1.0vox`. Область расчёта определялась исходной маской 00 с запасом 25 мм; отдельные целевые маски служили контролем. Средний прирост Dice относительно неподвижной маски составил **0,0282**; на всех проверенных узлах сохранённых полей `det J>0`, наименьшее измеренное значение — **0,410**. Это технический QC, а не независимая анатомическая валидация.

На фазе 10 прирост Dice всего **0,0001**. У фазы 03 после регистрации Dice остаётся около **0,901**, а 95-й процентиль расстояния перенесённых вершин до отдельной автоматической поверхности достигает **7,016 мм**. Эти отрицательные и ограничивающие наблюдения важны при дальнейшем использовании поля. Таблица всех фаз, параметры и хэши — в [пофазном отчёте](mp03_nix_full_cycle_report.md) и [агрегате](mp03_nix_full_cycle.json). Ранее отвергнутое поле 00→07 со 115 складками в общий ряд не включено; прежняя пара 00→01 пересчитана с единым более сильным сглаживанием.

Ни одно поле не получило **анатомического допуска**. Позднее четыре технических поля были применены только в [исследовательском парном FEM-сравнении](mp02_nix_paired_fem_report.md). Оно оценивает чувствительность модельного импеданса, а не истинность движения и не функции сердца.

## Фазовые маски TSv2 как контроль на полном разрешении

По решению автора отдельно полученные маски `whole_heart` TSv2 приняты как **автоматический контроль переноса фазы 00**. Для фаз 00, 01, 03, 07 и 10 проверены происхождение, бинарность, соответствие исходной КТ по сетке и аффинной матрице, а также хэши использованных в FEM масок. Вариант `products_v2` удаляет из прямого выхода TSv2 лишь небольшие отдельные фрагменты; он не добавляет вокселей. [Протокол фазового контроля](mp03_nix_ts2_reference_report.md) описывает эту обработку.

| Целевая фаза | Dice без переноса | Dice после Greedy | ASSD после Greedy, мм | HD95 после Greedy, мм |
|---|---:|---:|---:|---:|
| 01 | 0,93002 | 0,94897 | 0,98 | 3,75 |
| 03 | 0,85213 | 0,90099 | 1,88 | 7,30 |
| 07 | 0,92654 | 0,95905 | 0,83 | 2,76 |
| 10 | 0,96249 | 0,96259 | 0,72 | 2,50 |

Фаза 03 остаётся наиболее расходящейся, а выигрыш 10 практически отсутствует. Оценка поверхности вычислена по полным бинарным объёмам, поэтому её HD95 **7,30 мм** для фазы 03 не следует смешивать с **7,016 мм** для выборки перенесённых вершин в общем 11-фазном QC: это разные геометрические объекты и правила измерения.

Контрольная маска каждой фазы вычислена независимо от Greedy, но обе стороны сравнения автоматические. Их согласие не определяет абсолютную ошибку границы сердца. Совпадение хэшей с [парным FEM-опытом](mp02_nix_paired_fem_report.md) подтверждает происхождение входов, но не превращает TSv2 в ручной анатомический референс.

## Исторический пилот Георга: данные и масштаб

Ниже сохранён первоначальный двухпарный расчёт на 4D-КТ **Георга**. Его результаты описывают проверку метода до перехода к основному опыту Ника; переносить их численно на Ника нельзя. Исходная серия Георга содержит 15 фаз. Для пар 00→01 и 00→07 использованы КТ в единицах Хаунсфилда (HU) и автоматические маски `whole_heart` TSv2. Их исходная сетка `512×512×256`, шаг `0,458984×0,458984×0,625 мм`. Пространственная единица NIfTI обозначена `unknown`; RAS и миллиметры проверены по локальному манифесту конверсии и аффинной матрице. В ITK использованы физические координаты LPS, мм. SHA входов сохранены в [агрегате 01](mp03_04_georg_00_01_monai_qc.json) и [агрегате 07](mp03_04_georg_00_07_monai_qc.json).

Для раннего сравнения сформирована общая **уменьшенная** сетка с шагом около `1,377×1,377×1,25 мм`. На ней оценены неподвижный, жёсткий и деформируемый варианты. Позднейший расчёт на исходной сетке приведён отдельно ниже. Номер фазы не подтверждает ED/ES, точное время события или единый принятый R–R цикл; дыхание отдельно не установлено.

## Метод регистрации на уменьшенной сетке Георга

MONAI Physio версии `2026.09.1` вызвал модуль `picsl_greedy` в памяти на CPU. Для каждой пары `fixed` — целевая КТ, `moving` — фаза 00. Использованы аффинная инициализация, деформируемый шаг, метрика CC/WNCC, итерации `[12,6,3]` и сглаживание `2.0vox 0.5vox`. Маски определяли общий crop с запасом 20 мм и применялись для оценки; в подгонку поля они не входили.

Отображение `fixed_to_moving` ведёт физическую точку цели в источник и нужно для переноса значения исходной КТ или маски на сетку цели (pullback). Обратное `moving_to_fixed` ведёт вершину исходной поверхности в целевую фазу. Направления нельзя менять местами. Поле задано в LPS, мм; при переходе из NIfTI RAS изменены знаки координат x и y без изменения масштаба. Контроль композиции двух отображений сохранён в парных JSON. Код расчёта — [парный скрипт](mp03_04_monai_pair.py).

## Результат регистрации Георга на уменьшенной сетке

| Цель | Метод | Dice к отдельной маске | Ошибка центра, мм | Разница объёма, мл | ASSD / HD95, мм | CT-MSE, HU² |
|---|---|---:|---:|---:|---:|---:|
| 01 | неподвижный | 0,96013 | 0,884 | +17,773 | 0,902 / 2,754 | 5394 |
| 01 | жёсткий | 0,96013 | 0,884 | +17,773 | 0,902 / 2,754 | 4993 |
| 01 | Greedy | **0,97226** | **0,475** | **+2,725** | **0,637 / 1,860** | **1278** |
| 07 | неподвижный | 0,94869 | **0,595** | +29,198 | 1,145 / 3,750 | 7017 |
| 07 | жёсткий | 0,94869 | **0,595** | +29,198 | 1,145 / 3,750 | 6498 |
| 07 | Greedy | **0,96713** | 0,648 | **+2,266** | **0,722 / 2,314** | **1605** |

`CT-MSE` — средний квадрат разности интенсивностей КТ на области оценки; он измеряется в HU². Greedy уменьшил эту ошибку и расхождение автоматических масок в обеих парах. Для 07 ошибка центра **выросла** с 0,595 до 0,648 мм. Согласие с отдельной маской TSv2 не отделяет истинное движение от ошибки сегментации. Числа исходного разрешения сюда не добавлены из-за различия сеток.

## Контроль поля и повторяемости у Георга

Плотные поля `fixed_to_moving` хранятся локально в `runs/` как массивы с осями `z,y,x,component_LPS_xyz`; вектор смещения измеряется в миллиметрах. Контрольные суммы и физическая сетка записаны в парных агрегатах.

| Цель | `det J`: минимум / p01 / медиана / p99 / максимум | Доля `det J≤0` | P95 смещения, мм | Pullback вне исходного crop | Ошибка композиции: медиана / максимум, мм |
|---|---|---:|---:|---:|---:|
| 01 | 0,419 / 0,769 / 1,000 / 1,301 / 2,168 | 0 | 1,951 | 0,223% | 0,016 / 0,069 |
| 07 | 0,347 / 0,702 / 1,000 / 1,357 / 1,983 | 0 | 3,327 | 0,354% | 0,025 / 0,089 |

Неположительный `det J` не найден **на узлах уменьшенной сетки**. Проценты выхода относятся к crop, который доходит до верхней границы КТ, а не ко всему полю зрения. Повтор 00→01 с теми же параметрами дал близкие, но не одинаковые Dice (**0,97219** и **0,97226**) и минимумы якобиана (**0,445** и **0,419**); побитовая воспроизводимость программной реализации не установлена. Этот пилот не обосновывает отсутствие складок между узлами или на исходной сетке.

In [ ]:
# Проверка сохранённых агрегатов без чтения CT и масок.
import json
from pathlib import Path

for target in ("01", "07"):
    d = json.loads(Path(f"mp03_04_georg_00_{target}_monai_qc.json").read_text(encoding="utf-8"))
    assert d["software"]["monai_physio"] == "2026.09.1"
    assert d["fixed_target"] == f"phase_{target}"
    assert d["field_qc"]["jacobian_of_pullback"]["nonpositive_fraction"] == 0
    q = d["mask_qc"]
    print(target, "Dice", round(q["stationary"]["dice_vs_auto_mask"], 5),
          "->", round(q["greedy"]["dice_vs_auto_mask"], 5),
          "center_mm", round(q["greedy"]["center_difference_mm"], 3),
          "min_jac", round(d["field_qc"]["jacobian_of_pullback"]["minimum"], 3))

## Воспроизведение и решение для Георга

[Парный скрипт](mp03_04_monai_pair.py) принимает локальные каталоги КТ, масок, манифест конверсии и имена выходных JSON и полей. Запуски 00→01 и 00→07 выполняются отдельно; поля и изображения записываются только в исключённый `runs/`. Вычисления сохранённых агрегатов можно сверить кодовой ячейкой выше, не раскрывая медицинские массивы. Для полноразрешённого варианта параметры приведены в [отдельном отчёте](mp03_fullres_report.md).

Обе пары на уменьшенной сетке дали технически положительные поля, но независимого анатомического контура целевых фаз и принятой карты КТ Георга в его FEM-модель не было. Поэтому эти поля остались **кандидатами**, без допуска как индивидуальное движение сердца. Позднейший полноразрешённый опыт ниже проверяет часть прежних технических ограничений; он не заменяет анатомического референса.

## Регистрация Георга на исходном разрешении

Две пары Георга повторно зарегистрированы на исходном шаге `0,458984×0,458984×0,625 мм` без прореживания. Crop `414×441×235` определён по маске 00 с запасом 25 мм; целевая автоматическая маска служила только для внутренней оценки. [Полный отчёт](mp03_fullres_report.md) фиксирует команды, хэши и правила отбора поля.

| Пара и вариант | Dice к отдельной автоматической маске: неподвижно → поле | ASSD, мм: неподвижно → поле | CT-MSE, HU²: неподвижно → поле | Минимум `det J` / неположительных узлов |
|---|---:|---:|---:|---:|
| 00→01, исходное сглаживание | 0,95965 → 0,97043 | 0,732 → 0,502 | 4504 → 938 | 0,276 / 0 |
| 00→07, исходное сглаживание | 0,94898 → 0,96074 | 0,951 → 0,676 | 5088 → 1348 | −0,036 / 3 |
| 00→07, усиленное сглаживание | 0,94898 → 0,95950 | 0,951 → 0,702 | 5088 → 1596 | 0,466 / 0 |

Первый вариант 07 **отклонён из-за трёх складок** на проверенной сетке. Усиленное сглаживание устранило их на узлах, но несколько ухудшило Dice и CT-MSE относительно отвергнутого поля. В обоих технически допустимых по дискретному якобиану вариантах площадь ступенчатой перенесённой маски хуже согласуется с целевой автоматической маской, чем до переноса; эту площадь нельзя выдавать за гладкую анатомическую площадь. Кроме того, принятое преобразование КТ→FEM Георга и независимые ручные фазовые маски отсутствуют. Поля не стали валидированной геометрией для FEM или функций сердца.

In [ ]:
# Локальная сверка агрегатов; нужна копия исключённого из Git runs/.
import json
from pathlib import Path

for suffix in ("00_01", "00_07", "00_07_smooth"):
    d = json.loads(Path(f"runs/mp03_fullres_{suffix}_qc.json").read_text(encoding="utf-8"))
    assert d["grid_qc"]["stride_ijk"] == [1, 1, 1]
    assert d["grid_qc"]["crop_source_only"]
    print(suffix, d["mask_qc"]["greedy"]["dice_vs_auto_mask"],
          d["field_qc"]["jacobian_of_pullback"]["minimum"])
